# 03 — Recalibrate the DynaHug Oracle on this Corpus

**Purpose.** The upstream pretrained OCSVM collapses in this container environment (constant decision score). Recalibrate on the train half of the split (never the eval half), sweep hyperparameters, export a drop-in DYNAHUG_MODEL_DIR, and measure FP on the disjoint eval half.

**Inputs / outputs**
- `real_benign_corpus/oracle-split.json` (from notebook 02)
- `regenbench/dynahug` image
- `scripts/calibrate_oracle.py`, `fit_oracle_sweep.py`, `fp_eval_oracle.py`

**Outputs**
- `real_benign_corpus/oracle-calibrated/<ver>/` (model+vectorizer+scaler+report)
- eval-half FP report

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/check_oracle_disjointness.py"], check=False)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/calibrate_oracle.py", "real_benign_corpus/all_pt",
     "--split-file", "real_benign_corpus/oracle-split.json", "--split-role", "train",
     "--out", "real_benign_corpus/oracle-calibrated/pt",
     "--sample", "50", "--backend", "docker", "--seed", "1337", "--format", "pt"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/fit_oracle_sweep.py",
     "--traces", "real_benign_corpus/oracle-calibrated/current/traces.json",
     "--export", "--gamma", "0.1", "--nu", "0.01",
     "--export-dir", "real_benign_corpus/oracle-calibrated/current"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
show("real_benign_corpus/oracle-calibrated/current/calibration-report.json", max_lines=40)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/fp_eval_oracle.py", "--split", "real_benign_corpus/oracle-split.json",
     "--role", "eval", "--out", "real_benign_corpus/oracle-calibrated/current/fp-eval-eval.json", "--backend", "docker"], check=False)